# LightSeek-OCR — Training on Colab T4

**Before running:** `Runtime → Change runtime type → T4 GPU`

Checkpoints are saved to Google Drive so they survive session resets.

In [ ]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Mount Google Drive (checkpoints will be saved here) ──────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/lightseek-ocr/checkpoints'
DRIVE_METRICS_DIR    = '/content/drive/MyDrive/lightseek-ocr/metrics'

import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_METRICS_DIR,    exist_ok=True)
print(f'Checkpoints → {DRIVE_CHECKPOINT_DIR}')

In [ ]:
# ── 3. Clone repo (branch: feature/refactor) ────────────────────────────────
import os
REPO_DIR = '/content/lightseek-ocr'
BRANCH   = 'feature/refactor'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} https://github.com/Alfred0404/lightseek-ocr.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

In [ ]:
# ── 4. Install dependencies ─────────────────────────────────────────────────
# Colab ships with torch but may not have the latest cuda build — reinstall to be safe
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers peft accelerate datasets wonderwords jiwer

# Verify
import torch
print(f'torch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU ready: {torch.cuda.is_available()}')

In [ ]:
# ── 5. Patch train.py for T4 and Drive checkpoints ─────────────────────────
# We override the key constants directly here rather than editing train.py,
# so the repo stays clean.

import sys, os
sys.path.append('/content/lightseek-ocr/src')
sys.path.append('/content/lightseek-ocr/src/train')

# Config overrides (injected before importing the training function)
COLAB_CONFIG = {
    'DATASET_NAME'      : 'iam',   # 'iam' | 'cord' | 'synthetic'
    'BATCH_SIZE'        : 4,       # T4 has 16GB — safe at 4
    'ACCUMULATION_STEPS': 8,       # effective batch = 32
    'EPOCHS'            : 30,
    'MAX_TEXT_TOKENS'   : 128,
    'CHECKPOINT_DIR'    : DRIVE_CHECKPOINT_DIR,
    'METRICS_DIR'       : DRIVE_METRICS_DIR,
}
print('Config:', COLAB_CONFIG)

In [ ]:
# ── 6. Training loop (copy of train.py with Colab config injected) ──────────
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from LightSeekOCR import LightSeekOCR
from dataset import build_dataset
from prompt_template import TRANSCRIPTION_PROMPT
from utils.colors import bcolors


def collate_fn(batch):
    return [b[0] for b in batch], [b[1] for b in batch]


def plot_loss(epoch_losses, path):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(1, len(epoch_losses) + 1), epoch_losses,
            marker='o', color='royalblue', label='Train Loss')
    ax.set(title='Training Loss (live)', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(True)
    ax.annotate(f"{epoch_losses[-1]:.4f}",
                xy=(len(epoch_losses), epoch_losses[-1]),
                xytext=(8, 4), textcoords='offset points', fontsize=9, color='royalblue')
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close(fig)


def train(cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Training on {device}')

    model = LightSeekOCR(verbose=True)

    # Freeze SAM + CLIP
    for param in model.encoder.sam_extractor.model.parameters():
        param.requires_grad = False
    for param in model.encoder.clip_processor.model.parameters():
        param.requires_grad = False

    # SmolLM2 base frozen by LoRA; make encoder projectors trainable
    for param in model.encoder.compressor.parameters():
        param.requires_grad = True
    for param in model.encoder.channel_projection.parameters():
        param.requires_grad = True
    for param in model.decoder.visual_projection.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({trainable/total:.2%})')

    lora_params = [p for p in model.decoder.model.parameters() if p.requires_grad]
    optimizer = optim.AdamW([
        {'params': model.encoder.compressor.parameters(),         'lr': 1e-4},
        {'params': model.encoder.channel_projection.parameters(), 'lr': 1e-4},
        {'params': model.decoder.visual_projection.parameters(),  'lr': 1e-4},
        {'params': lora_params,                                   'lr': 5e-5},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['EPOCHS'])
    scaler    = GradScaler()

    dataset    = build_dataset(name=cfg['DATASET_NAME'], split='train')
    dataloader = DataLoader(dataset, batch_size=cfg['BATCH_SIZE'],
                            shuffle=True, collate_fn=collate_fn, num_workers=2)
    print(f'Dataset: {len(dataset)} samples')

    tokenizer  = model.decoder.tokenizer
    prompt_ids = tokenizer(TRANSCRIPTION_PROMPT, return_tensors='pt',
                           add_special_tokens=False).input_ids.to(device)
    N_prompt   = prompt_ids.shape[1]
    N_visual   = 512

    os.makedirs(cfg['CHECKPOINT_DIR'], exist_ok=True)
    os.makedirs(cfg['METRICS_DIR'],    exist_ok=True)
    loss_plot_path = os.path.join(cfg['METRICS_DIR'], 'loss_curve.png')

    epoch_losses = []
    step = 0

    for epoch in range(cfg['EPOCHS']):
        model.train()
        epoch_loss, n_samples = 0, 0
        optimizer.zero_grad()

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{cfg['EPOCHS']}")
        for images, texts in pbar:
            for image, text in zip(images, texts):
                with torch.no_grad():
                    features   = model.encoder.extract_features(image)
                compressed     = features['compressed_features']
                global_f       = features['global_features']
                local_f        = compressed.flatten(2).permute(0, 2, 1)

                text_ids = tokenizer(
                    text + tokenizer.eos_token,
                    return_tensors='pt', truncation=True,
                    max_length=cfg['MAX_TEXT_TOKENS']
                ).input_ids.to(device)

                combined_text_ids  = torch.cat([prompt_ids, text_ids], dim=1)
                combined_text_mask = torch.ones_like(combined_text_ids)

                labels = torch.cat([
                    torch.full((1, N_visual), -100, dtype=torch.long, device=device),
                    torch.full((1, N_prompt), -100, dtype=torch.long, device=device),
                    text_ids,
                ], dim=1)

                with autocast(device_type='cuda'):
                    outputs = model.decoder(
                        local_features=local_f, global_features=global_f,
                        text_input_ids=combined_text_ids,
                        text_attention_mask=combined_text_mask,
                        labels=labels,
                    )

                loss = outputs.loss / cfg['ACCUMULATION_STEPS']
                scaler.scale(loss).backward()
                epoch_loss += outputs.loss.item()
                n_samples  += 1
                step       += 1

                if step % cfg['ACCUMULATION_STEPS'] == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad], 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            pbar.set_postfix({'loss': f"{epoch_loss / max(n_samples,1):.4f}"})

        if step % cfg['ACCUMULATION_STEPS'] != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        scheduler.step()
        avg_loss = epoch_loss / max(n_samples, 1)
        epoch_losses.append(avg_loss)
        print(f'Epoch {epoch+1:02d}/{cfg["EPOCHS"]}  Loss: {avg_loss:.4f}')

        ckpt_path = os.path.join(cfg['CHECKPOINT_DIR'], f'model_epoch_{epoch+1}.pth')
        torch.save(model.state_dict(), ckpt_path)
        print(f'  Checkpoint → {ckpt_path}')
        plot_loss(epoch_losses, loss_plot_path)

    print(f'Training complete. Loss curve: {loss_plot_path}')


train(COLAB_CONFIG)

In [ ]:
# ── 7. Display loss curve ───────────────────────────────────────────────────
from IPython.display import Image as IPImage
import os
loss_plot = os.path.join(DRIVE_METRICS_DIR, 'loss_curve.png')
if os.path.exists(loss_plot):
    display(IPImage(loss_plot))
else:
    print('No loss curve yet — run training first.')